# Colab 收尾 2：檢查檔案、重新連接雲端硬碟、上傳、關機

In [1]:
# 1) 檢查：雲端硬碟掛載狀態、訓練結果實際在哪裡
!mount | grep -iE "drive|fuse" || echo "（沒有任何 drive 掛載）"
!ls /content/drive/MyDrive/MeetSum_finetune/run1 2>&1 | head
!find / -xdev \( -name "adapter_model.safetensors" -o -name "base_hf.json" -o -name "report.md" \) -size +0 2>/dev/null | grep -v "/proc/" | head -20
!du -sh /root/.config/Google/DriveFS 2>/dev/null; ls -la /content/ft_export 2>&1 | head

（沒有任何 drive 掛載）
ls: cannot access '/content/drive/MyDrive/MeetSum_finetune/run1': No such file or directory
5.0G	/root/.config/Google/DriveFS
total 16372132
drwxr-xr-x 2 root root        4096 Nov  6 11:28 .
drwxr-xr-x 1 root root        4096 Nov  6 14:15 ..
-rw-r--r-- 1 root root       11849 Nov  6 11:28 Modelfile
-rw-r--r-- 1 root root   217634848 Nov  6 11:06 typo-lora.gguf
-rw-r--r-- 1 root root 16547400192 Nov  6 11:28 typo-Q4_K_M.gguf


In [ ]:
# 2) 重新連接雲端硬碟（會跳出授權視窗）
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ls -la /content/drive/MyDrive/MeetSum_finetune/run1 /content/drive/MyDrive/MeetSum_finetune/results 2>&1 | head -40

In [ ]:
# 3) 把找到的結果複製進雲端硬碟（MeetSum_finetune/recovered/），再強制上傳
import glob, os, shutil, subprocess
dst = '/content/drive/MyDrive/MeetSum_finetune/recovered'
os.makedirs(dst, exist_ok=True)
found = subprocess.run('find / -xdev -path /proc -prune -o \\( -path "*run1/final/*" -o -path "*MeetSum_finetune/results/*" -o -path "*MeetSum_finetune/export/*" \\) -type f -print 2>/dev/null | grep -v "^/content/drive/"',
                       shell=True, capture_output=True, text=True).stdout.split()
for f in found:
    sub = 'final' if 'run1/final' in f else ('results' if '/results/' in f else 'export')
    os.makedirs(f'{dst}/{sub}', exist_ok=True)
    shutil.copy(f, f'{dst}/{sub}/{os.path.basename(f)}')
    print(f'{os.path.getsize(f)/1e6:9.1f} MB  {f}')
print('共', len(found), '個檔案')
drive.flush_and_unmount()
print('✅ 上傳完成')

In [ ]:
# 4) 關閉並釋放這台 Colab 機器（停止消耗運算單元）
from google.colab import runtime
runtime.unassign()